<a href="https://colab.research.google.com/github/tu702019/Machine_Learning_Algorithm_and_Its_Application/blob/main/HW1_bankruptcy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries & Data Information

In [72]:
# Import Libraries
import pandas as pd
from imblearn.combine import SMOTEENN
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.feature_selection import VarianceThreshold, SelectPercentile, f_classif

In [73]:
# load the data
url_bankruptcy = 'https://raw.githubusercontent.com/tu702019/Machine_Learning_Algorithm_and_Its_Application/refs/heads/main/HW1_Data%20Preprocessing/bankruptcy(predict%20brankrupt%20or%20not).csv'
df_bankruptcy = pd.read_csv(url_bankruptcy)

## Data Information

In [74]:
#Print the first 5 rows of the data
df_bankruptcy.head()

,Unnamed: 0,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0,1,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,NaN,0.564050,1.0,0.016469
1,1,1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1.0,0.020794
2,2,1,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1.0,0.016474
3,3,1,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1.0,0.023982
4,4,1,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,...,NaN,0.003878,NaN,0.598782,0.839973,0.278514,0.024752,0.575617,1.0,0.035490


In [75]:
# count the number of rows and coloumns in the dataset
df_bankruptcy.shape

(6819, 97)

In [76]:
df_bankruptcy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6819 entries, 0 to 6818
Data columns (total 97 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   Unnamed: 0                                                6819 non-null   int64  
 1   Bankrupt?                                                 6819 non-null   int64  
 2    ROA(C) before interest and depreciation before interest  6721 non-null   float64
 3    ROA(A) before interest and % after tax                   6719 non-null   float64
 4    ROA(B) before interest and depreciation after tax        6721 non-null   float64
 5    Operating Gross Margin                                   6719 non-null   float64
 6    Realized Sales Gross Margin                              6721 non-null   float64
 7    Operating Profit Rate                                    6719 non-null   float64
 8    Pre-tax net Inter

# Data Preprocessing

## Count the missing values

In [77]:
# Count the number of missing values in each column
missing_counts = df_bankruptcy.isnull().sum()

# Filter out columns that contain missing values
features_with_missing = missing_counts[missing_counts > 0]

print(f"Missing values are distributed across {len(features_with_missing)} features.")
print("Numbers of missing value:",df_bankruptcy.isnull().sum().sum())
print("Features with missing values:\n", features_with_missing)

Missing values are distributed across 95 features.
Numbers of missing value: 9440
Features with missing values:
 ROA(C) before interest and depreciation before interest     98
ROA(A) before interest and % after tax                     100
ROA(B) before interest and depreciation after tax           98
Operating Gross Margin                                     100
Realized Sales Gross Margin                                 98
                                                          ... 
Liability to Equity                                        100
Degree of Financial Leverage (DFL)                          99
Interest Coverage Ratio (Interest expense to EBIT)          99
Net Income Flag                                            100
Equity to Liability                                        100
Length: 95, dtype: int64


## Null/Missing Value Estimation

In [78]:
# Fill missing values based on data type
df_bankruptcy_filled = df_bankruptcy.copy()

for column in features_with_missing.index:
    # Categorical variables: Fill with mode
    if df_bankruptcy[column].dtype == 'object':
        df_bankruptcy_filled[column] = df_bankruptcy[column].fillna(df_bankruptcy[column].mode()[0])
    # Numerical variables: Fill with mean
    else:
        df_bankruptcy_filled[column] = df_bankruptcy[column].fillna(df_bankruptcy[column].mean())

In [79]:
# Drop unnecessary columns
df_bankruptcy_filled = df_bankruptcy_filled.drop(columns=['Unnamed: 0'], errors='ignore')

In [80]:
# check
print("numbers of missing value after filling:", df_bankruptcy_filled.isnull().sum().sum())

numbers of missing value after filling: 0


## Data Imbalance Processing

In [81]:
# Separate features and target
X = df_bankruptcy_filled.drop('Bankrupt?', axis=1)
y = df_bankruptcy_filled['Bankrupt?']

In [82]:
# Check class distribution before balancing
print("Class distribution before balancing:")
print(y.value_counts())
print(f"Class imbalance ratio: {y.value_counts()[0] / y.value_counts()[1]:.2f}")

Class distribution before balancing:
Bankrupt?
0    6599
1     220
Name: count, dtype: int64
Class imbalance ratio: 30.00


In [83]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [84]:
# Apply SMOTE+ENN
smote_enn = SMOTEENN(random_state=42)

# Apply the resampling
X_resampled, y_resampled = smote_enn.fit_resample(X_train, y_train)

In [85]:
# Check class distribution after balancing
print("\nClass distribution after SMOTE+ENN:")
print(pd.Series(y_resampled).value_counts())
print(f"Class balance ratio: {pd.Series(y_resampled).value_counts()[0] / pd.Series(y_resampled).value_counts()[1]:.2f}")


Class distribution after SMOTE+ENN:
Bankrupt?
1    4993
0    4285
Name: count, dtype: int64
Class balance ratio: 0.86


# Feature Selection

In [86]:
original_features_bankruptcy = X_resampled.shape[1]
print(f"Number of original features:", original_features_bankruptcy)

Number of original features: 95


In [87]:
# Step 1: Remove constant features (variance = 0)
vt_selector = VarianceThreshold(threshold=0.0)
X_resampled_filtered = vt_selector.fit_transform(X_resampled)
retained_columns = X.columns[vt_selector.get_support()]

# Step 2: Select top 30% features based on ANOVA F-value
selector = SelectPercentile(score_func=f_classif, percentile=30)
X_selected = selector.fit_transform(X_resampled_filtered, y_resampled)

# Get selected feature names
selected_features = retained_columns[selector.get_support()]
print("Selected features (top 30% based on F-score):")
print(selected_features.tolist())
print(f"Number of selected features: {len(selected_features)}")

Selected features (top 30% based on F-score):
[' ROA(C) before interest and depreciation before interest', ' ROA(A) before interest and % after tax', ' ROA(B) before interest and depreciation after tax', ' Operating Gross Margin', ' Realized Sales Gross Margin', ' Tax rate (A)', ' Net Value Per Share (B)', ' Net Value Per Share (A)', ' Net Value Per Share (C)', ' Persistent EPS in the Last Four Seasons', ' Operating Profit Per Share (Yuan ¥)', ' Per Share Net profit before tax (Yuan ¥)', ' Total Asset Return Growth Rate Ratio', ' Debt ratio %', ' Net worth/Assets', ' Operating profit/Paid-in capital', ' Net profit before tax/Paid-in capital', ' Operating profit per person', ' Working Capital to Total Assets', ' Quick Assets/Total Assets', ' Cash/Total Assets', ' Current Liability to Assets', ' Retained Earnings to Total Assets', ' CFO to Assets', ' Current Liability to Current Assets', ' Net Income to Total Assets', ' Gross Profit to Sales', ' Equity to Liability']
Number of selected f

# Save to Excel (.xlsx) & CSV (.csv)

In [88]:
# Step 1: Convert X_selected to DataFrame with column names
X_selected_df = pd.DataFrame(X_selected, columns=selected_features)

# Step 2: Combine with target variable
final_df = pd.concat([X_selected_df, pd.Series(y_resampled, name='Bankrupt?')], axis=1)

# Save to Excel (.xlsx)
final_df.to_excel("HW1_bankruptcy.xlsx", index=False)

# Save to CSV (.csv)
final_df.to_csv("HW1_bankruptcy.csv", index=False)

print("Saved to: HW1_bankruptcy.xlsx and HW1_bankruptcy.csv")


Saved to: HW1_bankruptcy.xlsx and HW1_bankruptcy.csv


# Valuation of Feature Selection

In [89]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Select features using f-classification selection
X_test_selected = X_test[selected_features]

# Model using all features
model_all = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42)
model_all.fit(X_resampled, y_resampled)

# Model using selected features
model_selected = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42)
model_selected.fit(X_selected_df, y_resampled)

# Evaluate model performance
print("Model with all features:")
print(classification_report(y_test, model_all.predict(X_test)))

print("Model with selected features:")
print(classification_report(y_test, model_selected.predict(X_test_selected)))


Model with all features:
              precision    recall  f1-score   support

           0       0.98      0.69      0.81      1320
           1       0.05      0.48      0.09        44

    accuracy                           0.69      1364
   macro avg       0.51      0.58      0.45      1364
weighted avg       0.95      0.69      0.79      1364

Model with selected features:
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1320
           1       0.19      0.91      0.31        44

    accuracy                           0.87      1364
   macro avg       0.59      0.89      0.62      1364
weighted avg       0.97      0.87      0.91      1364



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
